# 🩺 Skin Lesion Classification Pipeline

**Educational Demo: Transfer Learning for Medical Image Classification**

This notebook demonstrates the complete workflow for skin lesion classification 
using the PAD-UFES-20 dataset. It trains a **single backbone** for demonstration,
with pre-computed benchmark results from the full 5-fold CV experiment.

## 📋 Contents:
1. **Setup** - Environment detection & data loading
2. **EDA** - Class distribution visualization
3. **Data Prep** - Patient-level splitting (prevents data leakage)
4. **Dataset & Model** - DataLoader, backbone selection
5. **Training** - Two-stage transfer learning (frozen → fine-tuned)
6. **Evaluation** - Metrics, confusion matrix, classification report
7. **Benchmark** - Pre-computed results from full experiment (10 backbones × 5-fold CV)
8. **Grad-CAM** - Model explainability visualization

**Dataset:** [PAD-UFES-20 on Kaggle](https://www.kaggle.com/datasets/mahdavi1202/skin-cancer)  
**Full Project:** [GitHub Repository](https://github.com/muhwira27/skin-lesion-benchmark)

**Author:** Muh. Wira Ramdhani Fadhil

---

## 📥 1. Setup Emvironment

In [ ]:
# Check environment
import sys
import os

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

print(f"Running in Colab: {IN_COLAB}")
print(f"Running in Kaggle: {IN_KAGGLE}")

# Set base directories based on environment
if IN_KAGGLE:
    BASE_DIR = '/kaggle/working'
    DATA_DIR = '/kaggle/input/skin-cancer'
    # Kaggle has nested structure: imgs_part_1/imgs_part_1/
    IMG_DIRS = [
        f'{DATA_DIR}/imgs_part_1/imgs_part_1',
        f'{DATA_DIR}/imgs_part_2/imgs_part_2', 
        f'{DATA_DIR}/imgs_part_3/imgs_part_3'
    ]
    CSV_PATH = f'{DATA_DIR}/metadata.csv'
elif IN_COLAB:
    BASE_DIR = '/content'
    DATA_DIR = '/content/skin-cancer'
    IMG_DIRS = [
        f'{DATA_DIR}/imgs_part_1/imgs_part_1',
        f'{DATA_DIR}/imgs_part_2/imgs_part_2', 
        f'{DATA_DIR}/imgs_part_3/imgs_part_3'
    ]
    CSV_PATH = f'{DATA_DIR}/metadata.csv'
else:
    BASE_DIR = '.'
    DATA_DIR = 'data'
    IMG_DIRS = [f'{DATA_DIR}/images']
    CSV_PATH = f'{DATA_DIR}/metadata.csv'

print(f"Base directory: {BASE_DIR}")
print(f"Data directory: {DATA_DIR}")

### Download Dataset from Kaggle

In [ ]:
if IN_COLAB:
    from google.colab import files
    
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    
    kaggle_json = os.path.join(kaggle_dir, 'kaggle.json')
    if not os.path.exists(kaggle_json):
        print("Please upload your kaggle.json file:")
        uploaded = files.upload()
        for fn in uploaded.keys():
            os.rename(fn, kaggle_json)
        os.chmod(kaggle_json, 0o600)
    
    print("\nDownloading PAD-UFES-20 dataset...")
    os.makedirs(DATA_DIR, exist_ok=True)
    os.chdir(DATA_DIR)
    !kaggle datasets download -d mahdavi1202/skin-cancer --unzip
    os.chdir(BASE_DIR)
    print("✓ Dataset downloaded!")
    
elif IN_KAGGLE:
    print("Running on Kaggle - dataset available at:", DATA_DIR)
    print("\n📌 If dataset not found, add it manually:")
    print("   1. Click 'Add Input' button (top right)")
    print("   2. Search: 'mahdavi1202/skin-cancer'")
    print("   3. Click 'Add' to attach the dataset")
    print("   4. Re-run this cell")
else:
    print("Running locally - ensure data is in 'data/' folder")

### Import libraries

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### Configure Hyperparameters

In [ ]:
# Model
BACKBONE = 'resnet50'      # See backbone options below
IMG_SIZE = 224             # Image size (224 for most models)

# Data
TRAIN_RATIO = 0.70         # Train split ratio
VAL_RATIO = 0.15           # Validation split ratio  
TEST_RATIO = 0.15          # Test split ratio
BATCH_SIZE = 32            # Batch size

# Training
STAGE1_EPOCHS = 5          # Frozen backbone epochs
STAGE2_EPOCHS = 10         # Full fine-tuning epochs
LR_STAGE1 = 1e-3           # Learning rate (frozen)
LR_STAGE2 = 3e-4           # Learning rate (fine-tuning)
WEIGHT_DECAY = 0.01        # AdamW weight decay

# System
SEED = 42                  # Random seed
NUM_WORKERS = 0            # DataLoader workers (0 for Kaggle)

# ============================================================
# 🏆 BACKBONE OPTIONS (change BACKBONE above)
# ============================================================
# Rank | Name                   | F1    | Params
# -----|------------------------|-------|-------
#  1   | shufflenet_v2_x1_0     | 0.613 | 1.26M  ⭐ BEST
#  2   | vit_small_patch16_224  | 0.607 | 21.67M
#  3   | densenet121            | 0.604 | 6.96M
#  4   | seresnet50             | 0.594 | 26.05M
#  5   | tf_efficientnetv2_s    | 0.582 | 20.19M
#  6   | resnet50               | 0.579 | 23.52M
#  7   | regnety_032            | 0.576 | 17.93M
#  8   | mobilenet_v3_large     | 0.569 | 4.21M
#  9   | convnext_tiny          | 0.539 | 27.82M
# 10   | efficientnet_b0        | 0.504 | 4.02M
# ============================================================

## 📊 2. EDA

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Total images: {len(df)}, Patients: {df['patient_id'].nunique()}")
df.head()

In [ ]:
def find_image_path(img_id, img_dirs):
    for img_dir in img_dirs:
        path = os.path.join(img_dir, img_id)
        if os.path.exists(path):
            return path
    return None

# Class distribution
class_counts = df['diagnostic'].value_counts()
fig, ax = plt.subplots(figsize=(10, 5))
class_counts.plot(kind='bar', ax=ax, color=['#66BB6A', '#EF5350', '#D32F2F', '#66BB6A', '#EF5350', '#FFA726'])
ax.set_title('Class Distribution')
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 🔀 3. Data Preparation (Patient-Level Split)

In [ ]:
def create_patient_splits(df, train_r=TRAIN_RATIO, val_r=VAL_RATIO, test_r=TEST_RATIO, seed=SEED):
    """Patient-level stratified split to prevent data leakage."""
    patient_info = df.groupby('patient_id').agg({'diagnostic': lambda x: x.mode()[0]}).reset_index()
    patients, labels = patient_info['patient_id'].tolist(), patient_info['diagnostic'].tolist()
    
    train_val, test = train_test_split(patients, test_size=test_r, random_state=seed, stratify=labels)
    train_val_labels = [labels[patients.index(p)] for p in train_val]
    val_prop = val_r / (train_r + val_r)
    train, val = train_test_split(train_val, test_size=val_prop, random_state=seed, stratify=train_val_labels)
    
    return train, val, test

train_p, val_p, test_p = create_patient_splits(df)
train_df = df[df['patient_id'].isin(train_p)]
val_df = df[df['patient_id'].isin(val_p)]
test_df = df[df['patient_id'].isin(test_p)]

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

CLASSES = sorted(df['diagnostic'].unique())
label2id = {c: i for i, c in enumerate(CLASSES)}
id2label = {i: c for c, i in label2id.items()}

## 🏗️ 4. Dataset & Model

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def get_transforms(img_size=224, is_train=True):
    if is_train:
        return A.Compose([
            A.RandomResizedCrop(size=(img_size, img_size), scale=(0.8, 1.0)),  # v2.0+ API
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.5),
            A.ColorJitter(brightness=0.2, contrast=0.2, p=0.5),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(height=img_size, width=img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ])

class SkinDataset(Dataset):
    def __init__(self, df, img_dirs, label2id, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dirs = img_dirs
        self.label2id = label2id
        self.transform = transform
    
    def __len__(self): 
        return len(self.df)
    
    def find_img(self, img_id):
        """Find image across multiple directories."""
        for img_dir in self.img_dirs:
            path = os.path.join(img_dir, img_id)
            if os.path.exists(path):
                return path
        return None
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.find_img(row['img_id'])
        
        if img_path is None:
            raise FileNotFoundError(f"Image not found: {row['img_id']} in dirs: {self.img_dirs}")
        
        image = np.array(Image.open(img_path).convert('RGB'))
        if self.transform: 
            image = self.transform(image=image)['image']
        return image, self.label2id[row['diagnostic']]

# Verify image paths exist before creating datasets
print("Verifying image paths...")
sample_id = df.iloc[0]['img_id']

# Check which folders actually exist
valid_img_dirs = []
for d in IMG_DIRS:
    if os.path.exists(d):
        valid_img_dirs.append(d)
        print(f"  ✓ Found: {d} ({len(os.listdir(d))} files)")
    else:
        print(f"  ✗ Not found: {d}")

if not valid_img_dirs:
    # Try to auto-detect folder structure
    print("\n⚠️ No image folders found. Checking dataset structure...")
    if IN_KAGGLE:
        for item in os.listdir(DATA_DIR):
            item_path = os.path.join(DATA_DIR, item)
            if os.path.isdir(item_path) and 'img' in item.lower():
                valid_img_dirs.append(item_path)
                print(f"  Found: {item_path}")
    
if valid_img_dirs:
    IMG_DIRS = valid_img_dirs
    print(f"\n✅ Using image directories: {IMG_DIRS}")
else:
    raise ValueError("No valid image directories found!")

# Test loading one image
test_img = None
for d in IMG_DIRS:
    files = os.listdir(d)
    if files:
        test_path = os.path.join(d, files[0])
        test_img = Image.open(test_path)
        print(f"✅ Test image loaded: {test_path}")
        break

In [ ]:
train_ds = SkinDataset(train_df, IMG_DIRS, label2id, get_transforms(IMG_SIZE, True))
val_ds = SkinDataset(val_df, IMG_DIRS, label2id, get_transforms(IMG_SIZE, False))
test_ds = SkinDataset(test_df, IMG_DIRS, label2id, get_transforms(IMG_SIZE, False))

# Weighted sampler
weights = 1.0 / torch.tensor([train_df['diagnostic'].value_counts()[c] for c in CLASSES], dtype=torch.float)
sample_w = [weights[label2id[r['diagnostic']]] for _, r in train_df.iterrows()]
sampler = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

In [ ]:
# Build model (BACKBONE defined in hyperparameters at top)

def build_model(name, n_classes):
    """Build model from backbone name. Supports torchvision and timm models."""
    import torchvision.models as tv_models
    
    # Torchvision models
    if name == 'resnet50':
        model = tv_models.resnet50(weights='IMAGENET1K_V1')
        model.fc = nn.Linear(model.fc.in_features, n_classes)
    elif name == 'densenet121':
        model = tv_models.densenet121(weights='IMAGENET1K_V1')
        model.classifier = nn.Linear(model.classifier.in_features, n_classes)
    elif name == 'shufflenet_v2_x1_0':
        model = tv_models.shufflenet_v2_x1_0(weights='IMAGENET1K_V1')
        model.fc = nn.Linear(model.fc.in_features, n_classes)
    elif name == 'mobilenet_v3_large':
        model = tv_models.mobilenet_v3_large(weights='IMAGENET1K_V1')
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, n_classes)
    # timm models
    elif name in ['efficientnet_b0', 'tf_efficientnetv2_s', 'seresnet50', 
                  'vit_small_patch16_224', 'regnety_032', 'convnext_tiny']:
        model = timm.create_model(name, pretrained=True, num_classes=n_classes)
    else:
        # Fallback to timm for any other model
        print(f"Using timm for: {name}")
        model = timm.create_model(name, pretrained=True, num_classes=n_classes)
    
    return model

model = build_model(BACKBONE, len(CLASSES)).to(device)
print(f"\n✅ Model loaded: {BACKBONE}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

## 🔥 5. Training

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    loss_sum, preds, labels = 0, [], []
    for x, y in tqdm(loader, desc='Train'):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):  # Updated API
            out = model(x)
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item()
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(y.cpu().numpy())
    return loss_sum/len(loader), f1_score(labels, preds, average='macro')

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, preds, labels = 0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss_sum += criterion(out, y).item()
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(y.cpu().numpy())
    return loss_sum/len(loader), f1_score(labels, preds, average='macro'), balanced_accuracy_score(labels, preds), preds, labels

# Training setup (hyperparameters defined at top of notebook)

criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')
best_f1, best_state = 0, None

# Stage 1: Frozen backbone
for p in model.parameters(): p.requires_grad = False
for p in (model.fc if hasattr(model, 'fc') else model.head).parameters(): p.requires_grad = True
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_STAGE1)

print(f"Stage 1: Frozen backbone ({STAGE1_EPOCHS} epochs)")
for epoch in range(STAGE1_EPOCHS):
    tl, tf = train_epoch(model, train_loader, criterion, optimizer, scaler)
    vl, vf, va, _, _ = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{STAGE1_EPOCHS}: Train F1={tf:.4f}, Val F1={vf:.4f}")
    if vf > best_f1: best_f1, best_state = vf, model.state_dict().copy()

# Stage 2: Full fine-tuning
for p in model.parameters(): p.requires_grad = True
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_STAGE2)

print(f"\nStage 2: Full fine-tuning ({STAGE2_EPOCHS} epochs)")
for epoch in range(STAGE2_EPOCHS):
    tl, tf = train_epoch(model, train_loader, criterion, optimizer, scaler)
    vl, vf, va, _, _ = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{STAGE2_EPOCHS}: Train F1={tf:.4f}, Val F1={vf:.4f}")
    if vf > best_f1: best_f1, best_state = vf, model.state_dict().copy()

print(f"\n✅ Training complete! Best Val F1: {best_f1:.4f}")

## 📈 6. Evaluation

In [ ]:
model.load_state_dict(best_state)
_, test_f1, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion)
print(f"Test F1: {test_f1:.4f}, Balanced Acc: {test_acc:.4f}")
print(classification_report(test_labels, test_preds, target_names=CLASSES))

cm = confusion_matrix(test_labels, test_preds)
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(cmap='Blues')
plt.title(f'{BACKBONE} - F1: {test_f1:.4f}')
plt.show()

## 📊 7. Benchmark Results (Pre-computed)

⚠️ **Note:** The results below are **pre-computed** from the full **5-fold cross-validation** 
experiment using 10 different backbones. This notebook demonstrates training with a single 
backbone for educational purposes.

To reproduce the full benchmark, see the main repository:  
👉 [GitHub: skin-lesion-benchmark](https://github.com/muhwira27/skin-lesion-benchmark)

In [ ]:
print("📊 Pre-computed Benchmark Results (5-Fold Cross-Validation)")
print("=" * 60)
print("⚠️ These results are from the full experiment, not this notebook run.")
print()

results = pd.DataFrame({
    'Model': ['ShuffleNet V2', 'ViT-Small', 'DenseNet-121', 'SE-ResNet50', 
              'EfficientNetV2-S', 'ResNet-50', 'RegNetY-032', 'MobileNetV3', 
              'ConvNeXt', 'EfficientNet-B0'],
    'Macro-F1': [0.613, 0.607, 0.604, 0.594, 0.582, 0.579, 0.576, 0.569, 0.539, 0.504],
    'Std': [0.026, 0.027, 0.032, 0.035, 0.025, 0.022, 0.043, 0.060, 0.072, 0.089],
    'Params (M)': [1.26, 21.67, 6.96, 26.05, 20.19, 23.52, 17.93, 4.21, 27.82, 4.02]
})
results['Rank'] = range(1, len(results) + 1)
results = results[['Rank', 'Model', 'Macro-F1', 'Std', 'Params (M)']]
print(results.to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['gold', 'silver', '#CD7F32'] + ['steelblue'] * 7  # Gold, Silver, Bronze
ax.barh(results['Model'], results['Macro-F1'], color=colors)
ax.set_xlim(0.45, 0.65)
ax.set_xlabel('Macro F1 Score')
ax.set_title('Backbone Performance Comparison\n(5-Fold Cross-Validation, Pre-computed)')
ax.invert_yaxis()

# Add value labels
for i, (v, s) in enumerate(zip(results['Macro-F1'], results['Std'])):
    ax.text(v + 0.005, i, f'{v:.3f}±{s:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

# Key insights
print("\n💡 Key Insights:")
print(f"  🥇 Best: {results.iloc[0]['Model']} (F1={results.iloc[0]['Macro-F1']:.3f}, only {results.iloc[0]['Params (M)']:.2f}M params!)")
print(f"  📉 Worst: {results.iloc[-1]['Model']} (F1={results.iloc[-1]['Macro-F1']:.3f})")
print(f"  ⚠️ All models struggle with SCC class (recall ~20-30%)")

## ✅ Conclusion

- **Best model**: ShuffleNet V2 (1.26M params, F1=0.613)
- Patient-level splitting prevents data leakage
- Two-stage training improves transfer learning
- Full benchmark: [GitHub](https://github.com/muhwira27/skin-lesion-benchmark)

## 🧠 8. Explainability (Grad-CAM)

Custom implementation without external packages (works on Kaggle!)

In [ ]:
import cv2

class GradCAM:
    """Simple Grad-CAM implementation without external packages."""
    
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_backward_hook(self._save_gradient)
    
    def _save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        
        # Forward pass
        output = self.model(input_tensor)
        
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        
        # Backward pass
        self.model.zero_grad()
        output[0, target_class].backward()
        
        # Generate CAM
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        
        # Normalize
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        
        return cam.squeeze().cpu().numpy(), target_class

def get_target_layer(model, backbone_name):
    """Get the target layer for Grad-CAM."""
    if 'resnet' in backbone_name or 'seresnet' in backbone_name:
        return model.layer4[-1]
    elif 'densenet' in backbone_name:
        return model.features.denseblock4.denselayer16
    elif 'shufflenet' in backbone_name:
        return model.conv5[-1]
    elif 'mobilenet' in backbone_name:
        return model.features[-1]
    elif 'efficientnet' in backbone_name:
        return model.conv_head if hasattr(model, 'conv_head') else model.features[-1]
    else:
        # Default: try to find last conv layer
        last_conv = None
        for module in model.modules():
            if isinstance(module, nn.Conv2d):
                last_conv = module
        return last_conv

In [ ]:
# Visualize Grad-CAM on sample images
def visualize_gradcam_sample(model, test_df, img_dirs, backbone_name, device):
    """Visualize Grad-CAM for sample images."""
    
    target_layer = get_target_layer(model, backbone_name)
    gradcam = GradCAM(model, target_layer)
    
    # Get 3 sample images
    samples = []
    for cls in ['ACK', 'BCC', 'MEL']:
        sample = test_df[test_df['diagnostic'] == cls].iloc[0]
        img_path = find_image_path(sample['img_id'], img_dirs)
        if img_path:
            samples.append((img_path, cls))
    
    fig, axes = plt.subplots(len(samples), 3, figsize=(12, 4*len(samples)))
    
    for i, (img_path, true_label) in enumerate(samples):
        # Load image
        orig_img = Image.open(img_path).convert('RGB')
        img_array = np.array(orig_img)
        
        # Preprocess
        transform = get_transforms(224, is_train=False)
        input_tensor = transform(image=img_array)['image'].unsqueeze(0).to(device)
        
        # Get prediction
        model.eval()
        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1).cpu().numpy()[0]
            pred_idx = probs.argmax()
            pred_label = id2label[pred_idx]
            confidence = probs[pred_idx]
        
        # Generate Grad-CAM (need gradients, so no torch.no_grad)
        cam, _ = gradcam.generate(input_tensor, pred_idx)
        
        # Resize CAM to image size
        cam_resized = cv2.resize(cam, (224, 224))
        
        # Create heatmap
        img_resized = cv2.resize(img_array, (224, 224))
        heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        overlay = np.uint8(0.6 * img_resized + 0.4 * heatmap)
        
        # Plot
        axes[i, 0].imshow(orig_img)
        axes[i, 0].set_title(f'Original\nTrue: {true_label}')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(overlay)
        axes[i, 1].set_title(f'Grad-CAM\nPred: {pred_label} ({confidence*100:.1f}%)')
        axes[i, 1].axis('off')
        
        axes[i, 2].barh(CLASSES, probs, color=['green' if c == pred_label else 'gray' for c in CLASSES])
        axes[i, 2].set_xlim(0, 1)
        axes[i, 2].set_title('Probabilities')
    
    plt.suptitle(f'Grad-CAM Visualization ({backbone_name})', fontsize=14)
    plt.tight_layout()
    plt.show()

# Run visualization
print("Generating Grad-CAM visualizations...")
visualize_gradcam_sample(model, test_df, IMG_DIRS, BACKBONE, device)